In [2]:
# ============================================================
# 04_R_gpu_anvl.ipynb
# ANVL GPU benchmark for the Apollo Swiss route-choice MXL model
#
# This notebook evaluates an XLA-backed ANVL likelihood evaluator
# for the panel mixed-logit specification used in the Apollo Swiss
# route-choice example.
#
# Scope:
# - CUDA GPU execution through ANVL.
# - Deterministic Halton draws for simulated random coefficients.
# - Two optimisers: maxLik BFGS and BGW.
# - Repeated runs summarised by draw size to measure runtime variation.
#
# Main outputs:
# - results/raw/anvl_maxlik_gpu_*.rds
# - results/raw/anvl_bgw_gpu_*.rds
# - results/raw/anvl_maxlik_gpu_rep_results.csv
# - results/raw/anvl_bgw_gpu_rep_results.csv
# - results/benchmark_summary_anvl_gpu.csv
# - results/session_info_anvl_gpu.txt
# - results/anvl_gpu_results.zip
# ============================================================


In [3]:
# -------------------------------------------------------------------
# 1. Packages
# -------------------------------------------------------------------
# Packages are installed only when missing. The benchmark depends on
# ANVL for XLA-backed tensor evaluation, CUDA/PJRT for GPU execution,
# maxLik and BGW for optimisation, and randtoolbox for Halton draws.

apt_pkgs <- c(
  "build-essential", "cmake", "pkg-config",
  "libprotobuf-dev", "libprotoc-dev", "protobuf-compiler",
  "libuv1-dev", "libcurl4-openssl-dev", "libssl-dev", "libxml2-dev"
)

stopifnot(system2("apt-get", c("update", "-qq")) == 0L)
stopifnot(system2("apt-get", c("install", "-y", apt_pkgs)) == 0L)

# 2. R packages

Sys.setenv(
  CUDA_COMPONENTS = "nvshmem,nccl,nvcc",
  PJRT_INSTALL    = "1"
)

options(repos = c(
  "https://r-xla.r-universe.dev",
  "https://mlverse.r-universe.dev",
  "https://cloud.r-project.org"
))

pkgs <- c(
  "randtoolbox", "bgw", "maxLik",
  "cuda12.8", "pjrt", "anvl"
)

for (pkg in pkgs) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg)
  }
}

# 3. Load packages

suppressPackageStartupMessages({
  library(randtoolbox)
  library(bgw)
  library(maxLik)
  library(pjrt)
  library(anvl)
})


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘rngWELL’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘zoo’, ‘miscTools’, ‘sandwich’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘checkmate’, ‘safetensors’, ‘tengen’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘stablehlo’, ‘xlamisc’




In [5]:
# ============================================================
# Fast Apollo installation in Colab using Posit Package Manager
# ============================================================

options(HTTPUserAgent = sprintf(
  "R/%s R (%s)",
  getRversion(),
  paste(
    getRversion(),
    R.version["platform"],
    R.version["arch"],
    R.version["os"]
  )
))

# Auto-detect Ubuntu codename, e.g., jammy or noble
ubuntu_codename <- tryCatch(
  system("bash -lc '. /etc/os-release; echo ${VERSION_CODENAME}'", intern = TRUE),
  error = function(e) "jammy"
)

ppm_repo <- sprintf(
  "https://packagemanager.posit.co/cran/__linux__/%s/latest",
  ubuntu_codename[1]
)

options(repos = c(CRAN = ppm_repo))

cat("Using CRAN repo:\n", getOption("repos"), "\n\n")

install.packages("apollo", Ncpus = parallel::detectCores())

library(apollo)

pkg_versions <- data.frame(
  package = c("apollo", "randtoolbox", "bgw", "maxLik", "cuda12.8", "pjrt", "anvl"),
  version = vapply(
    c("apollo", "randtoolbox", "bgw", "maxLik", "cuda12.8", "pjrt", "anvl"),
    function(pkg) as.character(packageVersion(pkg)),
    character(1)
  )
)

print(pkg_versions)

Using CRAN repo:
 https://packagemanager.posit.co/cran/__linux__/jammy/latest 



Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘listenv’, ‘parallelly’, ‘future’, ‘globals’, ‘SparseM’, ‘MatrixModels’, ‘truncnorm’, ‘future.apply’, ‘mcmc’, ‘quantreg’, ‘mnormt’, ‘mvtnorm’, ‘numDeriv’, ‘Deriv’, ‘matrixStats’, ‘coda’, ‘Rsolnp’, ‘MCMCpack’, ‘RcppArmadillo’, ‘RcppEigen’




             . ,,                                                            
            ,      ,,                                                        
 ,,,,,,    ,         ,,                                                      
,     ,,  ,            ,,,,.                                                 
,,     , ,,   ,,,,,,    ,,,                                 //  //           
  ,     ,,,.   ,,,,,.   ,,      ////                        //  //           
,,     ,,,,,.           ,,     // //     //////    /////    //  //    /////  
,,,        ,,           ,      //  //    /    //  //   //   //  //   //   // 
              ,,  

                package version
apollo           apollo   0.3.9
randtoolbox randtoolbox   2.0.5
bgw                 bgw   0.1.4
maxLik           maxLik 1.5.2.2
cuda12.8       cuda12.8   1.0.9
pjrt               pjrt   0.5.0
anvl               anvl   0.4.0


In [6]:
# -------------------------------------------------------------------
# 2. Project-root resolution
# -------------------------------------------------------------------
# Run this notebook from the project root.

proj_root <- getwd()

raw_results_dir <- file.path(proj_root, "results", "raw")
summary_file <- file.path(proj_root, "results", "benchmark_summary_anvl_gpu.csv")
session_info_file <- file.path(proj_root, "results", "session_info_anvl_gpu.txt")

dir.create(raw_results_dir, recursive = TRUE, showWarnings = FALSE)


In [7]:
# ============================================================
# 3. Define benchmark settings
# ============================================================

# This section defines common settings and input builders for
# ANVL + maxLik + GPU and ANVL + BGW + GPU.

# ------------------------------------------------------------
# 3.1 Benchmark settings
# ------------------------------------------------------------

draws_grid <- c(500L, 1000L, 2000L, 5000L, 10000L)
seed <- 123L
# Standard deterministic Halton draws; seed is not used.
device_use <- "cuda"

# ------------------------------------------------------------
# 3.2 Load data and build ANVL database
# ------------------------------------------------------------
# Create ANVL data arrays with dimension task × individual × draw.

database <- apollo_swissRouteChoiceData
database <- database[order(database$ID), ]
row.names(database) <- NULL

n_indiv <- length(unique(database$ID))
n_task <- as.integer(nrow(database) / n_indiv)
n_obs <- nrow(database)

make_database_nv <- function() {

  make_x <- function(x) {
    nv_array(
      array(
        as.numeric(x),
        dim = c(n_task, n_indiv, 1L)
      ),
      dtype = "f64",
      device = device_use
    )
  }

  list(
    tt1 = make_x(database$tt1),
    tc1 = make_x(database$tc1),
    hw1 = make_x(database$hw1),
    ch1 = make_x(database$ch1),

    tt2 = make_x(database$tt2),
    tc2 = make_x(database$tc2),
    hw2 = make_x(database$hw2),
    ch2 = make_x(database$ch2),

    choice1 = make_x(database$choice == 1),
    choice2 = make_x(database$choice == 2)
  )
}

# ------------------------------------------------------------
# 3.3 Generate Halton normal draws
# ------------------------------------------------------------
# Create ANVL draw arrays with dimension task × individual × draw.

make_draws_nv <- function(num_draws) {

  halton_sequence <- halton(
    n   = n_indiv * num_draws,
    dim = 4
  )

  norm_value <- qnorm(halton_sequence)

  make_draw <- function(z) {

    draw_ind <- matrix(
      z,
      nrow = n_indiv,
      ncol = num_draws,
      byrow = TRUE
    )

    nv_array(
      array(
        as.numeric(draw_ind),
        dim = c(1L, n_indiv, num_draws)
      ),
      dtype = "f64",
      device = device_use
    )
  }

  list(
    tt = make_draw(norm_value[, 1]),
    tc = make_draw(norm_value[, 2]),
    hw = make_draw(norm_value[, 3]),
    ch = make_draw(norm_value[, 4])
  )
}

# ------------------------------------------------------------
# 3.4 Print benchmark information
# ------------------------------------------------------------

cat("Benchmark settings\n")
cat("Draw sizes   :", paste(draws_grid, collapse = ", "), "\n")
cat("Seed         :", seed, "\n")
cat("Device       :", device_use, "\n")
cat("Individuals  :", n_indiv, "\n")
cat("Tasks/person :", n_task, "\n")
cat("Observations :", n_obs, "\n")
cat("Raw output   :", raw_results_dir, "\n")
cat("Summary file :", summary_file, "\n")

# ------------------------------------------------------------
# 3.5 Verify GPU backend and save environment information
# ------------------------------------------------------------

cat("\nANVL device check:\n")
print(nv_array(1, dtype = "f64", device = device_use))

nvidia_smi <- tryCatch(
  system("nvidia-smi", intern = TRUE),
  error = function(e) paste("nvidia-smi failed:", conditionMessage(e))
)

cat("\nNVIDIA GPU check:\n")
cat(paste(nvidia_smi, collapse = "\n"), "\n")

sink(session_info_file)
cat("R session information\n")
print(sessionInfo())

cat("\nPackage versions\n")
for (pkg in c("apollo", "randtoolbox", "bgw", "maxLik", "cuda12.8", "pjrt", "anvl")) {
  cat(pkg, ": ", as.character(packageVersion(pkg)), "\n", sep = "")
}

cat("\nANVL device check\n")
print(nv_array(1, dtype = "f64", device = device_use))

cat("\nNVIDIA GPU information\n")
cat(paste(nvidia_smi, collapse = "\n"), "\n")
sink()

cat("Session and hardware information saved to:", session_info_file, "\n")


Benchmark settings
Draw sizes   : 500, 1000, 2000, 5000, 10000 
Seed         : 123 
Device       : cuda 
Individuals  : 388 
Tasks/person : 9 
Observations : 3492 
Raw output   : /content/results/raw 
Summary file : /content/results/benchmark_summary_anvl_gpu.csv 

ANVL device check:


<https://github.com/zml/pjrt-artifacts/releases/download/v14.0.1/pjrt-cuda_linux-amd64.tar.gz>


AnvlArray
 1
[ CUDAf64{1} ] 

NVIDIA GPU check:
Tue Aug 18 09:39:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P0             27W /   70W |   11299MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |


In [ ]:
# ============================================================
# 4. Define ANVL + maxLik estimation
# ============================================================

# This part of the code has been omitted temporarily and will be released upon publication of the corresponding paper.

In [ ]:
# ============================================================
# 5. Define ANVL + BGW likelihood function
# ============================================================

# This part of the code has been omitted temporarily and will be released upon publication of the corresponding paper.

In [10]:
# ============================================================
# 6. Run ANVL + maxLik GPU benchmark
# ============================================================

n_reps <- 5L

maxlik_rep_results <- list()
maxlik_results     <- list()

for (num_draws in draws_grid) {

  cat("\n============================================================\n")
  cat("ANVL + maxLik GPU | draws =", num_draws, "\n")
  cat("============================================================\n")

  # Build the maxLik-compatible likelihood and gradient.

  maxlik_functions <- make_maxlik_functions(
    num_draws = num_draws
  )

  logLik_fun <- maxlik_functions$logLik
  grad_fun   <- maxlik_functions$grad

  # ----------------------------------------------------------
  # 6.1 JIT compilation timing
  #
  # Includes compilation and the first evaluation of both the
  # likelihood and analytical gradient.
  # ----------------------------------------------------------

  jit_start <- proc.time()[["elapsed"]]

  ll_start_first <- logLik_fun(
    start_value_maxlik
  )

  grad_start_first <- grad_fun(
    start_value_maxlik
  )

  jit_time <- proc.time()[["elapsed"]] -
    jit_start

  # ----------------------------------------------------------
  # 6.2 Evaluation timing after JIT
  # ----------------------------------------------------------

  likelihood_start <- proc.time()[["elapsed"]]

  ll_start_second <- logLik_fun(
    start_value_maxlik
  )

  likelihood_time <- proc.time()[["elapsed"]] -
    likelihood_start

  gradient_start <- proc.time()[["elapsed"]]

  grad_start_second <- grad_fun(
    start_value_maxlik
  )

  gradient_time <- proc.time()[["elapsed"]] -
    gradient_start

  # Check that JIT compilation does not change the results.

  if (
    !isTRUE(
      all.equal(
        ll_start_first,
        ll_start_second
      )
    )
  ) {
    stop(
      "Likelihood check failed for draws = ",
      num_draws
    )
  }

  if (
    !isTRUE(
      all.equal(
        grad_start_first,
        grad_start_second
      )
    )
  ) {
    stop(
      "Gradient check failed for draws = ",
      num_draws
    )
  }

  # logLik_fun already returns the scalar total log-likelihood.

  initial_loglik <- sum(ll_start_second)

  cat("JIT time              :", jit_time, "seconds\n")
  cat("Likelihood eval. time :", likelihood_time, "seconds\n")
  cat("Gradient eval. time   :", gradient_time, "seconds\n")
  cat("Initial log-likelihood:", initial_loglik, "\n")

  # Store repetitions for the current number of draws.

  draw_rep_results <- vector(
    mode   = "list",
    length = n_reps
  )

  # ----------------------------------------------------------
  # 6.3 maxLik estimation repetitions
  # ----------------------------------------------------------

  for (rep_id in seq_len(n_reps)) {

    cat("\nRep", rep_id, "of", n_reps, "\n")

    # Count all likelihood and gradient calls made through
    # the maxLik interface.

    n_loglik_calls <- 0L
    n_grad_calls   <- 0L

    counted_logLik <- function(theta, ...) {

      n_loglik_calls <<- n_loglik_calls + 1L

      logLik_fun(
        theta,
        ...
      )
    }

    counted_grad <- function(theta, ...) {

      n_grad_calls <<- n_grad_calls + 1L

      grad_fun(
        theta,
        ...
      )
    }

    # Estimation timing excludes JIT compilation, final Hessian
    # calculation, console output and result-file saving.

    est_start <- proc.time()[["elapsed"]]

    result <- maxLik::maxLik(
      logLik = counted_logLik,
      grad   = counted_grad,
      start  = start_value_maxlik,
      method = "BFGS",

      # Do not calculate the final Hessian.

      finalHessian = FALSE,

      control = list(
        printLevel = 0L,
        iterlim    = 1000L
      )
    )

    estimation_time <- proc.time()[["elapsed"]] -
      est_start

    final_loglik <- as.numeric(
      stats::logLik(result)
    )

    # For maxLik's optim-based BFGS, code 0 indicates
    # successful convergence.

    if (result$code != 0L) {
      warning(
        "maxLik did not converge successfully for draws = ",
        num_draws,
        ", rep = ",
        rep_id,
        ". Return code: ",
        result$code,
        call. = FALSE
      )
    }

    # Saving is outside the estimation timer.

    result_file <- file.path(
      raw_results_dir,
      paste0(
        "anvl_maxlik_gpu_draws_",
        num_draws,
        "_rep_",
        rep_id,
        ".rds"
      )
    )

    saveRDS(
      result,
      result_file
    )

    rep_row <- data.frame(
      package                  = "ANVL",
      implementation           = "anvl_maxlik_gpu",
      backend                  = "GPU",
      optimiser                = "maxLik_BFGS",
      draws                    = num_draws,
      seed                     = seed,
      rep                      = rep_id,
      jit_time_sec             = jit_time,
      likelihood_eval_time_sec = likelihood_time,
      gradient_eval_time_sec   = gradient_time,
      initial_loglik           = initial_loglik,
      final_loglik             = final_loglik,
      estimation_time_sec      = estimation_time,
      convergence_code         = result$code,
      has_analytic_grad        = TRUE,
      n_loglik_eval            = n_loglik_calls,
      n_gradient_eval          = n_grad_calls,

      # Optimiser-reported objective-function evaluations.

      function_evals           = unname(result$iterations[1L]),
      result_file              = result_file
    )

    key <- paste(
      num_draws,
      rep_id,
      sep = "_"
    )

    draw_rep_results[[rep_id]] <- rep_row
    maxlik_rep_results[[key]]  <- rep_row

    cat("Final log-likelihood:", final_loglik, "\n")
    cat("Estimation time     :", estimation_time, "seconds\n")
    cat("Likelihood calls    :", n_loglik_calls, "\n")
    cat("Gradient calls      :", n_grad_calls, "\n")
    cat("Function evals      :", result$iterations[1L], "\n")
    cat("Convergence code    :", result$code, "\n")
  }

  # ----------------------------------------------------------
  # 6.4 Average over repetitions
  # ----------------------------------------------------------

  tmp <- do.call(
    rbind,
    draw_rep_results
  )

  maxlik_results[[as.character(num_draws)]] <- data.frame(
    package                   = "ANVL",
    implementation            = "anvl_maxlik_gpu",
    backend                   = "GPU",
    optimiser                 = "maxLik_BFGS",
    draws                     = num_draws,
    seed                      = seed,
    reps                      = n_reps,
    jit_time_sec              = jit_time,
    likelihood_eval_time_sec  = likelihood_time,
    gradient_eval_time_sec    = gradient_time,
    initial_loglik            = initial_loglik,
    mean_final_loglik         = mean(tmp$final_loglik),
    mean_estimation_time_sec  =
      mean(tmp$estimation_time_sec),
    sd_estimation_time_sec    =
      sd(tmp$estimation_time_sec),
    min_estimation_time_sec   =
      min(tmp$estimation_time_sec),
    max_estimation_time_sec   =
      max(tmp$estimation_time_sec),
    mean_n_loglik_eval        =
      mean(tmp$n_loglik_eval),
    mean_n_gradient_eval      =
      mean(tmp$n_gradient_eval),
    mean_function_evals       =
      mean(tmp$function_evals),
    all_repetitions_converged =
      all(tmp$convergence_code == 0L)
  )

  cat(
    "\nAverage estimation time:",
    mean(tmp$estimation_time_sec),
    "seconds\n"
  )

  cat(
    "Average likelihood calls:",
    mean(tmp$n_loglik_eval),
    "\n"
  )

  cat(
    "Average gradient calls  :",
    mean(tmp$n_gradient_eval),
    "\n"
  )

  cat(
    "Average function evals  :",
    mean(tmp$function_evals),
    "\n"
  )

  # Release data and device-backed arrays for this draw size.

  rm(
    maxlik_functions,
    logLik_fun,
    grad_fun,
    counted_logLik,
    counted_grad,
    ll_start_first,
    ll_start_second,
    grad_start_first,
    grad_start_second,
    draw_rep_results,
    rep_row,
    result,
    tmp
  )

  invisible(gc())
}


ANVL + maxLik GPU | draws = 500 
JIT time              : 2.934 seconds
Likelihood eval. time : 0.007 seconds
Gradient eval. time   : 0.009 seconds
Initial log-likelihood: -2253.778 

Rep 1 of 5 
Final log-likelihood: -1442.501 
Estimation time     : 0.625 seconds
Likelihood calls    : 92 
Gradient calls      : 32 
Function evals      : 91 
Convergence code    : 0 

Rep 2 of 5 
Final log-likelihood: -1442.501 
Estimation time     : 0.406 seconds
Likelihood calls    : 92 
Gradient calls      : 32 
Function evals      : 91 
Convergence code    : 0 

Rep 3 of 5 
Final log-likelihood: -1442.501 
Estimation time     : 0.413 seconds
Likelihood calls    : 92 
Gradient calls      : 32 
Function evals      : 91 
Convergence code    : 0 

Rep 4 of 5 
Final log-likelihood: -1442.501 
Estimation time     : 0.408 seconds
Likelihood calls    : 92 
Gradient calls      : 32 
Function evals      : 91 
Convergence code    : 0 

Rep 5 of 5 
Final log-likelihood: -1442.501 
Estimation time     : 0.417 sec

In [11]:
# ============================================================
# 7. Run ANVL + BGW GPU benchmark
# ============================================================

n_reps <- 5L
bgw_rep_results <- list()
bgw_results <- list()

for (num_draws in draws_grid) {

  cat("\n============================================================\n")
  cat("ANVL + BGW GPU | draws =", num_draws, "\n")
  cat("============================================================\n")

  calcRJ <- make_calcRJ_bgw(num_draws)
  calcR <- calcRJ$calcR
  calcJ <- calcRJ$calcJ

  # ----------------------------------------------------------
  # 7.1 JIT compilation timing
  # ----------------------------------------------------------

  jit_start <- proc.time()

  p_start_first <- calcR(start_value_bgw)
  J_start_first <- calcJ(start_value_bgw)

  jit_end <- proc.time()
  jit_time <- as.numeric((jit_end - jit_start)["elapsed"])

  # ----------------------------------------------------------
  # 7.2 Warm-up checks after JIT
  # ----------------------------------------------------------

  likelihood_start <- proc.time()
  p_start_second <- calcR(start_value_bgw)
  likelihood_end <- proc.time()

  likelihood_time <- as.numeric(
    (likelihood_end - likelihood_start)["elapsed"]
  )

  score_start <- proc.time()
  J_start_second <- calcJ(start_value_bgw)
  score_end <- proc.time()

  score_time <- as.numeric(
    (score_end - score_start)["elapsed"]
  )

  if (!isTRUE(all.equal(p_start_first, p_start_second))) {
    stop("Likelihood check failed for draws = ", num_draws)
  }

  if (!isTRUE(all.equal(J_start_first, J_start_second))) {
    stop("Score check failed for draws = ", num_draws)
  }

  initial_loglik <- sum(log(p_start_second))

  cat("JIT time              :", jit_time, "seconds\n")
  cat("Likelihood eval. time :", likelihood_time, "seconds\n")
  cat("Score eval. time      :", score_time, "seconds\n")
  cat("Initial log-likelihood:", initial_loglik, "\n")

  # ----------------------------------------------------------
  # 7.3 BGW estimation repetitions
  # ----------------------------------------------------------

  for (rep_id in seq_len(n_reps)) {

    cat("\nRep", rep_id, "of", n_reps, "\n")

    n_loglik_calls <- 0L
    n_score_calls <- 0L

    counted_calcR <- function(theta) {
      n_loglik_calls <<- n_loglik_calls + 1L
      calcR(theta)
    }

    counted_calcJ <- function(theta) {
      n_score_calls <<- n_score_calls + 1L
      calcJ(theta)
    }

    est_start <- proc.time()

    result <- bgw_mle(
      calcR = counted_calcR,
      betaStart = start_value_bgw,
      calcJ = counted_calcJ
    )

    est_end <- proc.time()

    estimation_time <- as.numeric(
      (est_end - est_start)["elapsed"]
    )

    if (!isTRUE(result$hasAnalyticGrad)) {
      stop("BGW did not use the analytical gradient.")
    }

    result_file <- file.path(
      raw_results_dir,
      paste0(
        "anvl_bgw_gpu_draws_",
        num_draws,
        "_rep_",
        rep_id,
        ".rds"
      )
    )

    saveRDS(result, result_file)

    key <- paste(num_draws, rep_id, sep = "_")

    bgw_rep_results[[key]] <- data.frame(
      package = "ANVL",
      implementation = "anvl_bgw_gpu",
      backend = "GPU",
      optimiser = "BGW",
      draws = num_draws,
      seed = seed,
      rep = rep_id,
      jit_time_sec = jit_time,
      likelihood_eval_time_sec = likelihood_time,
      score_eval_time_sec = score_time,
      initial_loglik = initial_loglik,
      final_loglik = result$finalLL,
      estimation_time_sec = estimation_time,
      convergence_code = result$code,
      has_analytic_grad = result$hasAnalyticGrad,
      n_loglik_eval = n_loglik_calls,
      n_score_eval = n_score_calls,
      function_evals = result$functionEvals,
      result_file = result_file
    )

    cat("Final log-likelihood:", result$finalLL, "\n")
    cat("Estimation time     :", estimation_time, "seconds\n")
    cat("Likelihood calls    :", n_loglik_calls, "\n")
    cat("Score calls         :", n_score_calls, "\n")
  }

  # ----------------------------------------------------------
  # 7.4 Average over repetitions
  # ----------------------------------------------------------

  tmp <- do.call(
    rbind,
    bgw_rep_results[
      grep(paste0("^", num_draws, "_"), names(bgw_rep_results))
    ]
  )

  bgw_results[[as.character(num_draws)]] <- data.frame(
    package = "ANVL",
    implementation = "anvl_bgw_gpu",
    backend = "GPU",
    optimiser = "BGW",
    draws = num_draws,
    seed = seed,
    reps = n_reps,
    jit_time_sec = jit_time,
    likelihood_eval_time_sec = likelihood_time,
    initial_loglik = initial_loglik,
    mean_final_loglik = mean(tmp$final_loglik),
    mean_estimation_time_sec = mean(tmp$estimation_time_sec),
    sd_estimation_time_sec = sd(tmp$estimation_time_sec),
    min_estimation_time_sec = min(tmp$estimation_time_sec),
    max_estimation_time_sec = max(tmp$estimation_time_sec),
    mean_n_loglik_eval = mean(tmp$n_loglik_eval)
  )

  cat(
    "\nAverage estimation time:",
    mean(tmp$estimation_time_sec),
    "seconds\n"
  )
  cat("Average likelihood calls:", mean(tmp$n_loglik_eval), "\n")
  cat("Average score calls     :", mean(tmp$n_score_eval), "\n")

  rm(
    calcRJ, calcR, calcJ,
    counted_calcR, counted_calcJ,
    p_start_first, p_start_second,
    J_start_first, J_start_second,
    result, tmp
  )

  gc()
}


ANVL + BGW GPU | draws = 500 
JIT time              : 1.652 seconds
Likelihood eval. time : 0.006 seconds
Score eval. time      : 0.008 seconds
Initial log-likelihood: -2253.778 

Rep 1 of 5 

BGW using analytic model derivatives supplied by caller...

There are no user-provided BGW settings. 
BGW settings have been set to default values...

            BetaName InitialBeta(i) D(i)
    1    mu_log_b_tt          -3.00    0
    2 sigma_log_b_tt           0.01    0
    3    mu_log_b_tc          -3.00    0
    4 sigma_log_b_tc           0.01    0
    5    mu_log_b_hw          -3.00    0
    6 sigma_log_b_hw           0.01    0
    7    mu_log_b_ch          -3.00    0
    8 sigma_log_b_ch           0.01    0


    it    nf     F            RELDF    PRELDF    RELDX    MODEL stppar   D*step   NPRELDF
     0     1 2.253777958e+03
     1     2 1.775420248e+03 2.122e-01 9.248e-03 1.97e-03   G   2.26e+01 9.30e-01 5.390e-02
     2     3 1.764574965e+03 6.109e-03 3.854e-02 8.47e-02   G   0.00e+00 

In [12]:
# ============================================================
# 8. Summarise outputs
# ============================================================

# ------------------------------------------------------------
# 8.1 Save repetition-level results
# ------------------------------------------------------------

maxlik_rep_summary <- do.call(
  rbind,
  maxlik_rep_results
)

bgw_rep_summary <- do.call(
  rbind,
  bgw_rep_results
)

row.names(maxlik_rep_summary) <- NULL
row.names(bgw_rep_summary) <- NULL

write.csv(
  maxlik_rep_summary,
  file.path(
    raw_results_dir,
    "anvl_maxlik_gpu_rep_results.csv"
  ),
  row.names = FALSE
)

write.csv(
  bgw_rep_summary,
  file.path(
    raw_results_dir,
    "anvl_bgw_gpu_rep_results.csv"
  ),
  row.names = FALSE
)

# ------------------------------------------------------------
# 8.2 Combine draw-level summaries
# ------------------------------------------------------------

maxlik_summary <- do.call(
  rbind,
  maxlik_results
)

bgw_summary <- do.call(
  rbind,
  bgw_results
)

# Keep only columns shared by both implementations.

common_cols <- intersect(
  names(maxlik_summary),
  names(bgw_summary)
)

benchmark_summary <- rbind(
  maxlik_summary[
    ,
    common_cols,
    drop = FALSE
  ],
  bgw_summary[
    ,
    common_cols,
    drop = FALSE
  ]
)

names(benchmark_summary)[
  names(benchmark_summary) == "mean_final_loglik"
] <- "final_loglik"

benchmark_summary <- benchmark_summary[
  order(
    benchmark_summary$draws,
    benchmark_summary$optimiser
  ),
]

row.names(benchmark_summary) <- NULL

# Check for unexpected missing values.

if (anyNA(benchmark_summary)) {
  stop("Unexpected NA found in the benchmark summary.")
}

# ------------------------------------------------------------
# 8.3 Add device information and save
# ------------------------------------------------------------

gpu_name <- tryCatch(
  suppressWarnings(
    system(
      paste(
        "nvidia-smi",
        "--query-gpu=name",
        "--format=csv,noheader"
      ),
      intern = TRUE,
      ignore.stderr = TRUE
    )
  ),
  error = function(e) character(0)
)

benchmark_summary$device <- if (length(gpu_name) > 0L) {
  trimws(gpu_name[1L])
} else {
  "Unknown"
}

other_cols <- setdiff(
  names(benchmark_summary),
  "device"
)

benchmark_summary <- benchmark_summary[
  ,
  append(
    other_cols,
    "device",
    after = match(
      "backend",
      other_cols
    )
  ),
  drop = FALSE
]

write.csv(
  benchmark_summary,
  summary_file,
  row.names = FALSE
)

print(benchmark_summary)

cat("\nSaved repetition-level results to:\n")

cat(
  file.path(
    raw_results_dir,
    "anvl_maxlik_gpu_rep_results.csv"
  ),
  "\n"
)

cat(
  file.path(
    raw_results_dir,
    "anvl_bgw_gpu_rep_results.csv"
  ),
  "\n"
)

cat("\nSaved benchmark summary to:\n")
cat(summary_file, "\n")

# ------------------------------------------------------------
# 8.4 Zip benchmark outputs
# ------------------------------------------------------------

zip_file <- file.path(
  proj_root,
  "results",
  "anvl_gpu_results.zip"
)

if (file.exists(zip_file)) {
  unlink(zip_file)
}

zip_files <- c(
  file.path(
    "results",
    "raw",
    list.files(raw_results_dir)
  ),
  file.path(
    "results",
    basename(summary_file)
  ),
  file.path(
    "results",
    basename(session_info_file)
  )
)

zip_files <- zip_files[
  file.exists(
    file.path(
      proj_root,
      zip_files
    )
  )
]

old_wd <- setwd(proj_root)

invisible(
  tryCatch(
    utils::zip(
      zipfile = zip_file,
      files = zip_files
    ),
    finally = setwd(old_wd)
  )
)

cat("\nSaved result zip to:\n")
cat(zip_file, "\n")

   package  implementation backend   device   optimiser draws seed reps
1     ANVL    anvl_bgw_gpu     GPU Tesla T4         BGW   500  123    5
2     ANVL anvl_maxlik_gpu     GPU Tesla T4 maxLik_BFGS   500  123    5
3     ANVL    anvl_bgw_gpu     GPU Tesla T4         BGW  1000  123    5
4     ANVL anvl_maxlik_gpu     GPU Tesla T4 maxLik_BFGS  1000  123    5
5     ANVL    anvl_bgw_gpu     GPU Tesla T4         BGW  2000  123    5
6     ANVL anvl_maxlik_gpu     GPU Tesla T4 maxLik_BFGS  2000  123    5
7     ANVL    anvl_bgw_gpu     GPU Tesla T4         BGW  5000  123    5
8     ANVL anvl_maxlik_gpu     GPU Tesla T4 maxLik_BFGS  5000  123    5
9     ANVL    anvl_bgw_gpu     GPU Tesla T4         BGW 10000  123    5
10    ANVL anvl_maxlik_gpu     GPU Tesla T4 maxLik_BFGS 10000  123    5
   jit_time_sec likelihood_eval_time_sec initial_loglik final_loglik
1         1.652                    0.006      -2253.778    -1445.690
2         2.934                    0.007      -2253.778    -1442.501
3